In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:

import os
import pandas as pd

path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(path)

print(f"Shape: {df.shape}") # the dataset shape

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:

df.info()

In [ ]:
# Task 4: Write your code here:

df.describe()

In [ ]:
# Task 1: Write your code here:

print("Missing values:")
print(df.isnull().sum())

In [ ]:

# i will drop it because it contains the most missing values
d= df.drop(columns=['D_142'])


for col in ['D_143', 'D_144', 'D_145', 'P_2']: # i wanted to to fill the missing values since im not sure these features are relevant to the code or not
    d[col] = d[col].fillna('unknown')




In [ ]:
print("Missing values remaining:", d.isnull().sum().sum())

In [ ]:
d.head()

In [ ]:
# Task 2: Write your code here:

def check_duplicates(d):
  duplicates = d.duplicated().sum()
  print(f"Number of Duplicate Samples: {duplicates}")
  if duplicates > 0:
    print("Dropping Duplicates...")
    d.drop_duplicates(inplace=True)
    print("Duplicates Dropped.")
  else:
    print("No Duplicate Samples Found.")

check_duplicates(d)

In [ ]:
# Task 3: Write your code here:

# NOT NEEDED since what we got from .info() are only numbers no objects/strings

In [ ]:
# Task 4: Write your code here:
from sklearn.model_selection import train_test_split, KFold
from sklearn.preprocessing import StandardScaler


numerical_cols = df.select_dtypes(include=["int64", "float64"]).columns.drop("Target")  # DON'T SCALE THE TARGET # IMP

scaler = StandardScaler()
df[numerical_cols] = scaler.fit_transform(df[numerical_cols])
df.head()


In [ ]:
# Task 5: Write your code here: # YES i didnt run the code but what i read from the data (the tearget column) it appears to have more 0 values than one indicating its imbalance

import seaborn as sns
import matplotlib.pyplot as plt

target_column =d["Target"]

def check_target_imbalance(d, target_column):
  print("Target Distribution:")
  print(df[target_column].value_counts(normalize=True))
  sns.countplot(x=df[target_column])
  plt.title("Target Distribution")
  plt.show()

check_target_imbalance(df, "Target")

In [ ]:

#task 1

X = df.drop("Target", axis=1).astype(float)
y = df['Target'].astype(float)


In [ ]:
from IPython.display import clear_output

%pip install kagglehub catboost xgboost tqdm -q

clear_output()

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score , f1_score
from catboost import CatBoostClassifier
import numpy as np

In [ ]:
# Task 2,3,4,5: Write your code here:

# because its considered imbalance i will use stratifiedKfold




sklearn_models = {

  "CatBoost": CatBoostClassifier(
      verbose=0,
      n_estimators=320,
      max_depth=4
  )
}

all_results = {}

for name in sklearn_models:
  all_results[name] = {'accuracy': [], 'precision': [], 'recall': [], 'f1': []}


n_splits = 5

skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)

for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):
  print(f"\nFold {fold_idx + 1}/{n_splits}")


  X_train, X_test = X.iloc[train_index], X.iloc[test_index]
  y_train, y_test = y.iloc[train_index], y.iloc[test_index]


  for model_name, model in sklearn_models.items():
    print(f"Training {model_name}...")
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)


    accuracy = accuracy_score(y_test, y_pred)

    f1 = f1_score(y_test, y_pred, zero_division=0)

    all_results[model_name]['accuracy'].append(accuracy)

    all_results[model_name]['f1'].append(f1)

    avg_loss = np.mean(all_results, axis=0) # PRINT AVERAGE ACROSS ALL FOLDS

In [ ]:
# Task 1: Write your code here:
# Gather importances from the models (from the last fold)
importances = {}

i
importances['CatBoost'] = sklearn_models['CatBoost'].feature_importances_

# Create a 1x3 plot
fig, axes = plt.subplots(1, 3, figsize=(18, 6))
axes = axes.flatten()
features = X.columns

for i, (model_name, imp) in enumerate(importances.items()):
  # Sort features by importance for a cleaner plot
  sorted_idx = np.argsort(imp)

  ax = axes[i]
  ax.barh(features[sorted_idx], imp[sorted_idx])
  ax.set_title(f"{model_name} Feature Importance")
  ax.set_xlabel("Importance Score")

plt.tight_layout()
plt.show()


In [ ]:
# Task 2: Write your code here:


In [ ]:
# Task Bonus: Write your code here: